# Assignment 4: Sales and Student Performance Analysis

## Objective of the Assignment
1. Find the total number of products sold in each category using PySpark.
2. Perform multivariate analysis on the Iris dataset using PCA and clustering.
3. Count students in each grade for every subject and visualize marks using PCA.

Run the cells in order. Sales records and student marks are sample data made for this assignment.


In [ ]:
%pip -q install pyspark pandas numpy scikit-learn matplotlib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from IPython.display import display

spark = SparkSession.builder.appName("Assignment4").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark started successfully")

## 1. Find the quantity sold in each product category

We use a small **sample sales dataset** created in the code. Each row contains a product category, quantity ordered, and order status.

Only orders with the status **Shipped** are counted. We add the quantities for each product category using PySpark.


In [ ]:
# Sample sales records: (Product Category, Quantity Ordered, Order Status)
sales_data = [
    ("Electronics", 12, "Shipped"),
    ("Clothing", 20, "Shipped"),
    ("Stationery", 35, "Shipped"),
    ("Home Decor", 10, "Shipped"),
    ("Electronics", 8, "Pending"),
    ("Clothing", 15, "Shipped"),
    ("Stationery", 18, "Cancelled"),
    ("Sports", 14, "Shipped"),
    ("Home Decor", 9, "Shipped"),
    ("Electronics", 7, "Shipped"),
    ("Sports", 11, "Pending"),
    ("Clothing", 10, "Cancelled"),
    ("Stationery", 22, "Shipped"),
    ("Sports", 6, "Shipped"),
    ("Home Decor", 16, "Pending"),
    ("Electronics", 5, "Shipped"),
    ("Clothing", 13, "Shipped"),
    ("Sports", 9, "Shipped"),
]

sales = spark.createDataFrame(sales_data, ["PRODUCTLINE", "QUANTITYORDERED", "STATUS"])

# Take only shipped orders.
shipped = sales.filter(F.col("STATUS") == "Shipped")

# Add up the quantities for each category.
category_sales = (shipped.groupBy("PRODUCTLINE")
                  .agg(F.sum("QUANTITYORDERED").alias("Total_Quantity"))
                  .orderBy(F.desc("Total_Quantity")))

category_sales.show()
sales_result = category_sales.toPandas()
sales_result.to_csv("category_sales.csv", index=False)


In [ ]:
# Bar chart of total quantity sold in each category.
plt.figure(figsize=(9, 4))
plt.barh(sales_result["PRODUCTLINE"], sales_result["Total_Quantity"])
plt.xlabel("Quantity Sold")
plt.ylabel("Product Category")
plt.title("Category-wise Sales")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 2. Multivariate analysis using the Iris dataset

We use the **Iris dataset** from scikit-learn. It contains 150 flower samples with four measurements: sepal length, sepal width, petal length, and petal width.

First, we standardize the data. Then we use **PCA** to keep at least 90% of the variance and **hierarchical clustering** to group similar flowers.

We compare 2 to 5 clusters using the silhouette score. The actual flower species are shown for comparison.


In [ ]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

# Load and standardize the Iris dataset.
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
X_scaled = StandardScaler().fit_transform(X)

# Keep enough principal components to explain 90% variance.
pca = PCA(n_components=0.90)
X_pca = pca.fit_transform(X_scaled)

# Try different numbers of clusters.
results = []
for k in range(2, 6):
    model = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels = model.fit_predict(X_pca)
    score = silhouette_score(X_pca, labels)
    results.append([k, score])

score_table = pd.DataFrame(results, columns=["Clusters", "Silhouette_Score"])
display(score_table)

# Select the cluster count with the highest silhouette score.
best_k = int(score_table.loc[score_table["Silhouette_Score"].idxmax(), "Clusters"])
cluster_model = AgglomerativeClustering(n_clusters=best_k, linkage="ward")
clusters = cluster_model.fit_predict(X_pca)

print("Number of samples:", len(X))
print("Number of original features:", X.shape[1])
print("Principal components kept:", pca.n_components_)
print("Variance explained:", round(sum(pca.explained_variance_ratio_) * 100, 2), "%")
print("Best number of clusters:", best_k)

# Average measurement values for each cluster.
cluster_data = X.copy()
cluster_data["Cluster"] = clusters + 1
display(cluster_data.groupby("Cluster").mean().round(2))

# Compare clusters with actual flower species.
display(pd.crosstab(clusters + 1, iris.target + 1,
                    rownames=["Cluster"], colnames=["Iris Species"]))

In [ ]:
# Plot cumulative variance explained by PCA.
variance = np.cumsum(pca.explained_variance_ratio_)
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(variance) + 1), variance, marker="o")
plt.axhline(0.90, linestyle="--", color="gray")
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA Explained Variance")
plt.show()

# Show the clusters using the first two components.
plt.figure(figsize=(7, 4))
for i in range(best_k):
    plt.scatter(X_pca[clusters == i, 0], X_pca[clusters == i, 1],
                label=f"Cluster {i + 1}")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("Iris Clusters using PCA")
plt.legend()
plt.show()

iris_result = pd.DataFrame({"Cluster": clusters + 1,
                            "PC1": X_pca[:, 0], "PC2": X_pca[:, 1]})
iris_result.to_csv("iris_groups.csv", index=False)

The **silhouette score** helps us understand how well the flower samples are grouped. A higher score usually means the groups are better separated. The scatter plot displays the first two principal components, while clustering uses all PCA components kept.


## 3. Count students in each grade for every subject

We use **sample marks for 24 students** in five subjects: DBMS, Networks, Statistics, Algorithms, and Python. These are made-up marks for practice, not actual student results.

To use an actual CSV, enter its filename in `RESULTS_CSV`. The file should have a `student_id` column and subject columns with marks from 0 to 100.

Grade ranges: S (90–100), A (80–89), B (70–79), C (60–69), D (50–59), E (40–49), F (below 40).


In [ ]:
RESULTS_CSV = ""  # Example: "my_marks.csv"

if RESULTS_CSV:
    marks = pd.read_csv(RESULTS_CSV)
else:
    # Create the same sample marks every time.
    np.random.seed(42)
    student_level = np.linspace(35, 90, 24).reshape(24, 1)
    sample_marks = np.clip(np.round(student_level + np.random.normal(0, 10, (24, 5))), 0, 100)
    marks = pd.DataFrame(sample_marks,
                         columns=["DBMS", "Networks", "Statistics", "Algorithms", "Python"])
    marks.insert(0, "student_id", range(1, 25))
    print("Using sample marks for 24 students")

subjects = marks.columns.drop("student_id").tolist()
marks[subjects] = marks[subjects].apply(pd.to_numeric)

# Convert numerical marks into grades.
def get_grade(mark):
    if mark >= 90:
        return "S"
    elif mark >= 80:
        return "A"
    elif mark >= 70:
        return "B"
    elif mark >= 60:
        return "C"
    elif mark >= 50:
        return "D"
    elif mark >= 40:
        return "E"
    else:
        return "F"

# Find each student's average and overall grade.
student_summary = pd.DataFrame()
student_summary["student_id"] = marks["student_id"]
student_summary["Average"] = marks[subjects].mean(axis=1)
student_summary["Grade"] = student_summary["Average"].apply(get_grade)

display(student_summary.round(2))

In [ ]:
# Convert marks into subject-grade pairs.
subject_grade_pairs = []
for subject in subjects:
    for mark in marks[subject]:
        subject_grade_pairs.append((subject, get_grade(mark)))

# Count each pair using Spark RDD (MapReduce approach).
rdd = spark.sparkContext.parallelize(subject_grade_pairs)
grade_counts = (rdd.map(lambda x: (x, 1))
                  .reduceByKey(lambda a, b: a + b)
                  .collect())

# Make a table of grade counts for each subject.
count_data = pd.DataFrame([(subject, grade, count)
                           for (subject, grade), count in grade_counts],
                          columns=["Subject", "Grade", "Students"])
grade_order = ["S", "A", "B", "C", "D", "E", "F"]
subject_counts = count_data.pivot(index="Subject", columns="Grade", values="Students")
subject_counts = subject_counts.reindex(index=subjects, columns=grade_order).fillna(0).astype(int)

overall_counts = student_summary["Grade"].value_counts().reindex(grade_order, fill_value=0)

print("Grades for each subject:")
display(subject_counts)
print("Overall student grades:")
display(overall_counts.to_frame("Students"))

subject_counts.to_csv("subject_grade_counts.csv")
student_summary.to_csv("student_grades.csv", index=False)

In the Spark MapReduce step, each `(subject, grade)` pair gets a count of 1. `reduceByKey()` adds these counts to find how many students received each grade in each subject.

PCA is used separately to visualize patterns in student marks. It does not calculate the grades.

In [ ]:
# Apply PCA to student marks.
mark_values = marks[subjects]
scaled_marks = StandardScaler().fit_transform(mark_values)
marks_pca = PCA(n_components=2)
student_points = marks_pca.fit_transform(scaled_marks)

print("Variance explained by two components:",
      round(sum(marks_pca.explained_variance_ratio_) * 100, 2), "%")

# Plot subject-wise grade counts.
subject_counts.plot(kind="bar", stacked=True, figsize=(9, 4))
plt.xlabel("Subjects")
plt.ylabel("Number of Students")
plt.title("Grades in Each Subject")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

# Plot student marks in two PCA dimensions.
plt.figure(figsize=(7, 4))
for grade in grade_order:
    selected = student_summary["Grade"] == grade
    if selected.any():
        plt.scatter(student_points[selected, 0], student_points[selected, 1], label=grade)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("Student Marks using PCA")
plt.legend(title="Overall Grade")
plt.show()

spark.stop()
print("Analysis completed. CSV result files have been saved.")

## Conclusion

1. **Sales analysis:** PySpark calculates the total quantity sold for each product category using shipped orders.
2. **Iris analysis:** PCA reduces the number of measurements, and hierarchical clustering groups flowers based on their features.
3. **Student analysis:** Spark counts grades for every subject. PCA gives a simple two-dimensional view of student marks.

The CSV files created by the notebook contain the results of each analysis.